# Gaussian MLE: gradient-based hyperparameter learning

> **Status:** placeholder — content coming soon.

[Notebook 05](05_gaussian_mcmc.ipynb) inferred the tree's hyperparameters with **MCMC** — a full Bayesian posterior, calibrated uncertainty, but $O(N_{\text{chain}})$ BFFG evaluations to get there. This notebook covers the **point-estimate cousin**: maximum-likelihood (or MAP) estimation by gradient descent through the BFFG marginal likelihood.

The headline is a v3-specific feature: every `SweepFn` is a **pure `Tree → Tree` function** with `Tree` registered as a JAX pytree. That means the whole `bffg_cycle` — `init_sde_leaves → sde_up → propagate_v_T_to_v_0 → sde_down_conditional` — is `jax.grad`-differentiable end-to-end. We can compute

$$\nabla_\theta\, \log p(\text{obs} \mid \theta)$$

directly and feed it to **optax** for SGD / Adam. v2's mutable `tree.data` made this impossible; v3 was designed for it.

## Planned outline

1. Same Gaussian linear-tree model and synthetic dataset as [notebook 04](04_phylo_bayesian.ipynb) / [05](05_gaussian_mcmc.ipynb)
2. Define the negative log-likelihood: `nll(theta) = -bffg_root_loglik(tree, theta)`
3. `jax.grad(nll)` — verify it returns finite gradients on log-transformed parameters
4. **optax** training loop (Adam, ~300 steps) starting from overdispersed init — should converge to the same posterior mode that MCMC finds
5. Comparison plot: MLE point estimates vs MCMC posteriors from notebook 05
6. Side-bar: **MAP** = MLE + log-prior — one-line change in the objective

## Why MLE matters here

MCMC and MLE are two solutions to the same question. MLE is cheaper (1 forward + 1 backward pass per step vs MCMC's many) and is a good warm-up for MCMC (initialise the chain at the MLE → fewer burn-in iterations). And because v3's tree IS differentiable, MLE costs only what one BFFG cycle plus its `grad` costs — i.e. *the same order* as one MCMC step. v2 simply could not do this — the mutable `tree.data` broke `jax.grad` straight away.